In [1]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 1 — Imports, Paths, and Global Constants
# ═══════════════════════════════════════════════════════════════════════
!pip install -q timm

import torch, torch.nn as nn
import torchvision.transforms as T
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import LambdaLR
import pandas as pd, numpy as np, json, time, random, itertools
from PIL import Image
from pathlib import Path
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score
import timm
import warnings; warnings.filterwarnings("ignore")

DATA_ROOT = Path("/kaggle/input/datasets/ashery/chexpert")
TRAIN_CSV  = DATA_ROOT / "train.csv"
OUTPUT_DIR = Path("/kaggle/working"); OUTPUT_DIR.mkdir(exist_ok=True)

MODEL_NAME        = "vit_b16"
IMAGE_SIZE        = 224    # ViT-B/16 native resolution (16x16 patches → 14x14=196 patches)
NUM_CLASSES       = 14
DEVICE            = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEED              = 42
HP_SEARCH_SUBSET  = 0.05   # 5% for ViT — 86M params needs more GPU memory per trial
HP_SEARCH_EPOCHS  = 5
FINAL_EPOCHS      = 15
HP_PATIENCE       = 3
FINAL_PATIENCE    = 5
MAX_TRIALS        = 36     # Random search (Bergstra & Bengio, 2012)

LABEL_NAMES = [
    "No Finding","Enlarged Cardiomediastinum","Cardiomegaly","Lung Opacity",
    "Lung Lesion","Edema","Consolidation","Pneumonia","Atelectasis",
    "Pneumothorax","Pleural Effusion","Pleural Other","Fracture","Support Devices"
]
torch.manual_seed(SEED); np.random.seed(SEED); random.seed(SEED)
print(f"Device: {DEVICE}  |  Model: {MODEL_NAME}  |  Image: {IMAGE_SIZE}x{IMAGE_SIZE}")

Device: cuda  |  Model: vit_b16  |  Image: 224x224


In [3]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 3 — CheXpert Dataset + Transforms [FIXED]
# ═══════════════════════════════════════════════════════════════════════
from PIL import Image
import numpy as np

def csv_to_abs_path(raw_csv_path, data_root):
    rel = raw_csv_path.replace("CheXpert-v1.0-small/", "")
    return Path(data_root) / rel

def filter_existing_files(df, data_root):
    print("  Building valid patient index from disk (fast scan)...")
    train_dir = Path(data_root) / "train"
    valid_dir = Path(data_root) / "valid"
    existing_patients = set()
    for d in [train_dir, valid_dir]:
        if d.exists():
            for p in d.iterdir():
                if p.is_dir():
                    existing_patients.add(p.name)
    def patient_exists(raw_path):
        parts = raw_path.replace("CheXpert-v1.0-small/", "").split("/")
        return parts[1] if len(parts) > 1 else None
    mask = df["Path"].apply(lambda p: patient_exists(p) in existing_patients)
    n_missing = (~mask).sum()
    print(f"  ✓ {mask.sum():,} valid  |  ✗ {n_missing:,} missing (dropped)")
    return df[mask].reset_index(drop=True)

class CheXpertDataset(Dataset):
    def __init__(self, df, image_root, transform=None, uncertain_policy="ones"):
        self.df = df.reset_index(drop=True)
        self.image_root = Path(image_root)
        self.transform = transform
        self.df[LABEL_NAMES] = self.df[LABEL_NAMES].fillna(0)
        self.df[LABEL_NAMES] = self.df[LABEL_NAMES].replace(
            -1, 1 if uncertain_policy == "ones" else 0)

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row      = self.df.iloc[idx]
        img_path = csv_to_abs_path(row["Path"], self.image_root)
        img      = Image.open(img_path).convert("RGB")
        lbl      = torch.tensor(row[LABEL_NAMES].values.astype(np.float32))
        if self.transform: img = self.transform(img)
        return img, lbl

def get_transforms(train=True):
    if train:
        return T.Compose([T.Resize((IMAGE_SIZE, IMAGE_SIZE)),
                          T.RandomHorizontalFlip(0.5), T.RandomRotation(15),
                          T.RandomAffine(degrees=0, translate=(0.1, 0.1)),
                          T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1),
                          T.ToTensor(),
                          T.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])])
    return T.Compose([T.Resize((IMAGE_SIZE, IMAGE_SIZE)), T.ToTensor(),
                      T.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])])

# ── Load, filter, split ───────────────────────────────────────────────
print("Loading CheXpert data...")
full_df = pd.read_csv(TRAIN_CSV)
print(f"  Raw CSV rows: {len(full_df):,}")

full_df = filter_existing_files(full_df, DATA_ROOT)

hp_df, _         = train_test_split(full_df, test_size=1-HP_SEARCH_SUBSET, random_state=SEED)
hp_train, hp_val = train_test_split(hp_df,   test_size=0.15,               random_state=SEED)
print(f"  HP train: {len(hp_train):,}  |  HP val: {len(hp_val):,}  |  Full (clean): {len(full_df):,}")

Loading CheXpert data...


  Raw CSV rows: 223,414
  Building valid patient index from disk (fast scan)...


  ✓ 223,414 valid  |  ✗ 0 missing (dropped)
  HP train: 9,494  |  HP val: 1,676  |  Full (clean): 223,414


In [4]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 4 — ViT-B/16 Model + Optimizer Builder
# ═══════════════════════════════════════════════════════════════════════
class CheXpertViT(nn.Module):
    """ViT-B/16 with custom classifier head — tunable dropout."""
    def __init__(self, hp, pretrained=True):
        super().__init__()
        self.backbone   = timm.create_model("vit_base_patch16_224", pretrained=pretrained, num_classes=0)
        embed_dim       = self.backbone.embed_dim  # 768 for ViT-Base
        drop            = float(hp["dropout"])
        self.classifier = nn.Sequential(nn.LayerNorm(embed_dim), nn.Dropout(drop),
                                        nn.Linear(embed_dim, NUM_CLASSES))
    def forward(self, x):
        return self.classifier(self.backbone(x))

def build_model(hp):
    return CheXpertViT(hp, pretrained=True)

def build_opt_sched(model, hp, total_epochs=HP_SEARCH_EPOCHS):
    wu = int(hp["warmup_epochs"])
    opt = AdamW(model.parameters(), lr=float(hp["learning_rate"]), weight_decay=float(hp["weight_decay"]))
    # Warmup + cosine decay — critical for ViT stability (Dosovitskiy et al., 2020)
    def lr_lambda(epoch):
        if epoch < wu:   return (epoch + 1) / wu           # linear warmup
        return 0.5 * (1 + np.cos(np.pi * (epoch - wu) / max(total_epochs - wu, 1)))
    sch = LambdaLR(opt, lr_lambda=lr_lambda)
    return opt, sch

In [5]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 5 — Class Weight Computation
# ═══════════════════════════════════════════════════════════════════════
def compute_pos_weights(loader):
    pos, total = torch.zeros(NUM_CLASSES), 0
    for _, lbl in loader:
        pos += lbl.sum(0); total += lbl.size(0)
    return (total - pos) / (pos + 1e-5)

In [6]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 6 — Train / Validate Functions (with Gradient Clipping for ViT)
# ═══════════════════════════════════════════════════════════════════════
def train_one_epoch(model, loader, criterion, optimizer, scaler, grad_clip):
    model.train(); total_loss = 0
    for imgs, lbls in loader:
        imgs, lbls = imgs.to(DEVICE), lbls.to(DEVICE)
        optimizer.zero_grad()
        with torch.amp.autocast("cuda"):
            loss = criterion(model(imgs), lbls)
        scaler.scale(loss).backward()
        # Gradient clipping — essential for ViT: prevents exploding gradients [V3]
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
        scaler.step(optimizer); scaler.update()
        total_loss += loss.item()
    return total_loss / len(loader)

def validate_epoch(model, loader, criterion):
    model.eval(); total_loss, preds, labels = 0, [], []
    with torch.no_grad():
        for imgs, lbls in loader:
            imgs, lbls = imgs.to(DEVICE), lbls.to(DEVICE)
            with torch.amp.autocast("cuda"):
                logits = model(imgs); loss = criterion(logits, lbls)
            total_loss += loss.item()
            preds.append(torch.sigmoid(logits).cpu().numpy())
            labels.append(lbls.cpu().numpy())
    p = np.concatenate(preds); l = np.concatenate(labels)
    aurocs = [roc_auc_score(l[:,i],p[:,i]) for i in range(NUM_CLASSES) if l[:,i].sum()>0]
    return total_loss/len(loader), np.mean(aurocs), p, l

In [8]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 8 — Hardcoded Optimal Config (from completed HP search)
# Trial 18 → AUROC 0.7491 | lr=3e-5, bs=16, wd=1e-4, dropout=0.1,
#             warmup_epochs=2, grad_clip=2.0
# ═══════════════════════════════════════════════════════════════════════

optimal = {
    "learning_rate"  : 3e-5,
    "batch_size"     : 16,
    "weight_decay"   : 1e-4,
    "dropout"        : 0.1,
    "warmup_epochs"  : 2,
    "grad_clip"      : 2.0,
}
val_auroc_phase1 = 0.7491

print("Optimal HP loaded:")
for k, v in optimal.items():
    print(f"  {k:<20} {v}")
print(f"  Phase 1 best AUROC   {val_auroc_phase1}")


Optimal HP loaded:
  learning_rate        3e-05
  batch_size           16
  weight_decay         0.0001
  dropout              0.1
  warmup_epochs        2
  grad_clip            2.0
  Phase 1 best AUROC   0.7491


In [9]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 9 — Final Training
# ═══════════════════════════════════════════════════════════════════════
print(f"\n{'='*68}\n  PHASE 2 — FINAL TRAINING  |  {MODEL_NAME.upper()}  |  {FINAL_EPOCHS} epochs\n{'='*68}")
opt_hp = {k:v for k,v in optimal.items() if k!="val_auroc_phase1"}

full_train, full_val = train_test_split(full_df, test_size=0.05, random_state=SEED)
bs_f  = int(opt_hp["batch_size"])
ft_ds = CheXpertDataset(full_train, DATA_ROOT, get_transforms(True))
fv_ds = CheXpertDataset(full_val,   DATA_ROOT, get_transforms(False))
ft_ld = DataLoader(ft_ds, bs_f, shuffle=True,  num_workers=4, pin_memory=True)
fv_ld = DataLoader(fv_ds, bs_f, shuffle=False, num_workers=4, pin_memory=True)

final_model = build_model(opt_hp).to(DEVICE)
final_pos_w = compute_pos_weights(ft_ld).to(DEVICE)
final_crit  = nn.BCEWithLogitsLoss(pos_weight=final_pos_w)
final_opt, final_sch = build_opt_sched(final_model, opt_hp, FINAL_EPOCHS)
final_scaler = torch.amp.GradScaler("cuda")
ckpt_path    = OUTPUT_DIR / f"best_{MODEL_NAME}.pth"
final_grad_clip = float(opt_hp["grad_clip"])

final_log, best_auroc, best_epoch, patience = [], 0.0, 0, 0
for ep in range(1, FINAL_EPOCHS+1):
    tr_loss              = train_one_epoch(final_model, ft_ld, final_crit, final_opt, final_scaler, final_grad_clip)
    vl_loss, auroc, _,_  = validate_epoch(final_model, fv_ld, final_crit)
    cur_lr = final_opt.param_groups[0]["lr"]
    if final_sch: final_sch.step()
    wu_tag = " [WARMUP]" if ep <= int(opt_hp["warmup_epochs"]) else ""
    final_log.append({"epoch":ep,"train_loss":round(tr_loss,5),"val_loss":round(vl_loss,5),
                      "val_auroc":round(auroc,5),"lr":round(cur_lr,8)})
    marker = ""
    if auroc > best_auroc:
        best_auroc, best_epoch, patience = auroc, ep, 0
        torch.save({"epoch":ep,"model_state_dict":final_model.state_dict(),
                    "auroc":auroc,"optimal_hp":opt_hp,"history":final_log}, ckpt_path)
        marker = " ✓ NEW BEST"
    else:
        patience+=1; marker=f" ({patience}/{FINAL_PATIENCE})"
        if patience>=FINAL_PATIENCE: print(f"  ⚠ Early stop epoch {ep}"); break
    print(f"  Ep{ep:>2}/{FINAL_EPOCHS} | TrLoss:{tr_loss:.4f} VlLoss:{vl_loss:.4f} AUROC:{auroc:.4f} LR:{cur_lr:.2e}{wu_tag}{marker}")
pd.DataFrame(final_log).to_csv(OUTPUT_DIR/f"final_log_{MODEL_NAME}.csv", index=False)
print(f"\nPhase 2 done | Best AUROC: {best_auroc:.4f} at epoch {best_epoch}")


  PHASE 2 — FINAL TRAINING  |  VIT_B16  |  15 epochs


model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

  Ep 1/15 | TrLoss:0.9480 VlLoss:0.9016 AUROC:0.7722 LR:1.50e-05 [WARMUP] ✓ NEW BEST


  Ep 2/15 | TrLoss:0.9189 VlLoss:0.9100 AUROC:0.7721 LR:3.00e-05 [WARMUP] (1/5)


  Ep 3/15 | TrLoss:0.8954 VlLoss:0.8802 AUROC:0.7807 LR:3.00e-05 ✓ NEW BEST


...

  Ep 6/15 | TrLoss:0.8522 VlLoss:0.8649 AUROC:0.7930 LR:2.62e-05 ✓ NEW BEST


  Ep 7/15 | TrLoss:0.8350 VlLoss:0.8845 AUROC:0.7922 LR:2.35e-05 (1/5)


  Ep 8/15 | TrLoss:0.8142 VlLoss:0.8882 AUROC:0.7932 LR:2.03e-05 ✓ NEW BEST


  Ep 9/15 | TrLoss:0.7871 VlLoss:0.9212 AUROC:0.7964 LR:1.68e-05 ✓ NEW BEST


  Ep10/15 | TrLoss:0.7509 VlLoss:0.9380 AUROC:0.7935 LR:1.32e-05 (1/5)


In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# CELL 10 — Per-Label Evaluation
# ═══════════════════════════════════════════════════════════════════════
ckpt = torch.load(ckpt_path, weights_only=False)
final_model.load_state_dict(ckpt["model_state_dict"])
_, _, fp, fl = validate_epoch(final_model, fv_ld, final_crit)
print(f"\n{'='*68}\n  FINAL PER-LABEL RESULTS — {MODEL_NAME.upper()}\n{'='*68}")
per_label = []
for i, name in enumerate(LABEL_NAMES):
    if fl[:,i].sum()==0: continue
    auroc = roc_auc_score(fl[:,i], fp[:,i])
    ap    = average_precision_score(fl[:,i], fp[:,i])
    f1    = f1_score(fl[:,i], (fp[:,i]>=0.5).astype(int), zero_division=0)
    per_label.append({"label":name,"auroc":round(auroc,4),"ap":round(ap,4),"f1":round(f1,4)})
    print(f"  {name:<30}  {auroc:>7.4f}  {ap:>7.4f}  {f1:>7.4f}")
ma = np.mean([r["auroc"] for r in per_label])
print(f"\n  MEAN AUROC: {ma:.4f}")
pd.DataFrame(per_label).to_csv(OUTPUT_DIR/f"per_label_{MODEL_NAME}.csv", index=False)

print(f"\n{'='*68}")
print(f"  ALL OUTPUT FILES — download from Kaggle Output tab")
print(f"{'='*68}")
for fname in [f"hp_log_{MODEL_NAME}.json", f"hp_summary_{MODEL_NAME}.csv",
              f"sensitivity_{MODEL_NAME}.txt", f"optimal_{MODEL_NAME}.json",
              f"final_log_{MODEL_NAME}.csv", f"per_label_{MODEL_NAME}.csv",
              f"best_{MODEL_NAME}.pth"]:
    print(f"  ✓  {fname}")
print(f"{'='*68}")